In [0]:
%sql
select
  account_id,
  trim(account_code) as account_code,
  trim(account_name) as account_name,
  trim(account_type) as account_type,
  trim(category) as category,
  coalesce(is_active, true) as is_active
from mini_project.bronze.accounts
where account_id is not null
  and account_code is not null
  and account_name is not null

In [0]:
%sql
select * from mini_project.bronze.general_ledgers

In [0]:
%sql
select * from mini_project.bronze.payroll

In [0]:
%sql
select * from mini_project.bronze.employee

In [0]:
%sql
create or replace table mini_project.silver.silver_employee (
  employee_id int,
  employee_code string,
  first_name string,
  last_name string,
  email string,
  department_id int,
  company_id int,
  position string,
  join_date date,
  termination_date date,
  base_salary double,
  is_active boolean
);

insert into mini_project.silver.silver_employee
select
  cast(employee_id as int) as employee_id,
  employee_code,
  first_name,
  last_name,
  email,
  department_id,
  company_id,
  position,
  to_date(hire_date, 'dd-MM-yyyy HH:mm') as join_date,
  to_date(termination_date, 'dd-MM-yyyy HH:mm') as termination_date,
  round(cast(base_salary as decimal(18,2)), 2) as base_salary,
  is_active
from mini_project.bronze.employee;

In [0]:
%sql
select
base_salary
from mini_project.silver.silver_employee limit 100

In [0]:
%sql
UPDATE mini_project.silver.silver_employee
SET base_salary = ROUND(base_salary, 2);

#accounts 


In [0]:
%sql
select * from mini_project.bronze.accounts

In [0]:
%sql
select distinct account_name,account_type,category from mini_project.bronze.accounts;
select distinct account_type from mini_project.bronze.accounts;
select distinct category from mini_project.bronze.accounts;

#company

In [0]:
%sql
select * from mini_project.bronze.company

#departments


In [0]:
%sql
select * from mini_project.bronze.departments

#gl

In [0]:
select * from mini_project.silver.silver_general_ledgers

In [0]:
CREATE OR REPLACE TABLE mini_project.silver.silver_general_ledgers (
  gl_id INT,
  entry_date DATE,
  posting_date DATE,
  fiscal_year INT,
  fiscal_period INT,
  company_id INT,
  account_id INT,
  department_id INT,
  transaction_type STRING,
  reference_number STRING,
  description STRING,
  debit_amount DOUBLE,
  credit_amount DOUBLE,
  created_by STRING,
  approved_by STRING,
  status STRING
);

INSERT into mini_project.silver.silver_general_ledgers
SELECT
  CAST(gl_id AS INT) AS gl_id,
  to_date(entry_date, 'dd-MM-yyyy HH:mm') AS entry_date,
  to_date(posting_date, 'dd-MM-yyyy HH:mm') AS posting_date,
  CAST(fiscal_year AS INT) AS fiscal_year,
  CAST(fiscal_period AS INT) AS fiscal_period,
  CAST(company_id AS INT) AS company_id,
  CAST(account_id AS INT) AS account_id,
  CAST(department_id AS INT) AS department_id,
  CAST(transaction_type AS STRING) AS transaction_type,
  CAST(reference_number AS STRING) AS reference_number,
  CAST(description AS STRING) AS description,
  ROUND(CAST(debit_amount AS DOUBLE), 2) AS debit_amount,
  ROUND(CAST(credit_amount AS DOUBLE), 2) AS credit_amount,
  CAST(created_by AS STRING) AS created_by,
  CAST(approved_by AS STRING) AS approved_by,
  CAST(status AS STRING) AS status
FROM mini_project.bronze.general_ledgers;